# 🎮 Full Fine-tuning v1 (Game-optimized / KcELECTRA)

**보정된 UnSmile 데이터만** 사용하여 학습합니다.

- **모델**: `beomi/KcELECTRA-base-v2022`
- **메트릭**: `abuse_recall`
- **방식**: Full Fine-tuning (모든 파라미터 학습)

In [1]:
import os, torch, pandas as pd, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import precision_recall_fscore_support, label_ranking_average_precision_score
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available(): print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA: True
GPU: NVIDIA L40S


In [2]:
MODEL_NAME = "beomi/KcELECTRA-base-v2022"
OUTPUT_DIR = "./output/full_game_kcelectra"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 통일된 하이퍼파라미터 (Full FT는 LR 낮춤)
EPOCHS, BATCH_SIZE, LEARNING_RATE = 5, 16, 2e-5
MAX_LENGTH = 128

LABEL_NAMES = ["여성/가족", "남성", "성소수자", "인종/국적", "연령", "지역", "종교", "기타 혐오", "악플/욕설", "clean"]
NUM_LABELS = len(LABEL_NAMES)

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
train_df = pd.read_csv("../../3_UnSmile_Correction/unsmile_train_corrected.tsv", sep='\t')
valid_df = pd.read_csv("../../3_UnSmile_Correction/unsmile_valid_corrected.tsv", sep='\t')
print(f"✅ Train: {len(train_df)}건, Valid: {len(valid_df)}건")

✅ Train: 14690건, Valid: 3663건


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    tokenized = tokenizer(examples['문장'], padding='max_length', truncation=True, max_length=MAX_LENGTH)
    tokenized['labels'] = [[float(examples[col][i]) for col in LABEL_NAMES] for i in range(len(examples['문장']))]
    return tokenized

train_dataset = Dataset.from_pandas(train_df).map(preprocess_function, batched=True, remove_columns=train_df.columns.tolist())
valid_dataset = Dataset.from_pandas(valid_df).map(preprocess_function, batched=True, remove_columns=valid_df.columns.tolist())

Map:   0%|          | 0/14690 [00:00<?, ? examples/s]

Map:   0%|          | 0/3663 [00:00<?, ? examples/s]

In [5]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS, problem_type="multi_label_classification").to(DEVICE)
print(f"Total params: {sum(p.numel() for p in model.parameters()):,} (100% trainable)")

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at beomi/KcELECTRA-base-v2022 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total params: 127,784,458 (100% trainable)


In [6]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(predictions)).numpy()
    preds = (probs > 0.5).astype(int)
    labels_int = labels.astype(int)
    _, abuse_r, abuse_f1, _ = precision_recall_fscore_support(labels_int[:,8], preds[:,8], average='binary', zero_division=0)
    _, clean_r, clean_f1, _ = precision_recall_fscore_support(labels_int[:,9], preds[:,9], average='binary', zero_division=0)
    return {'lrap': label_ranking_average_precision_score(labels, predictions), 'abuse_recall': abuse_r, 'abuse_f1': abuse_f1, 'clean_recall': clean_r, 'clean_f1': clean_f1}

In [7]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, warmup_ratio=0.1, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="abuse_recall", greater_is_better=True,
    logging_steps=50, save_total_limit=2, report_to="none", fp16=True, gradient_accumulation_steps=2
)
trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=valid_dataset,
                  tokenizer=tokenizer, compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])

In [8]:
print("🚀 Full FT 학습 시작...")
trainer.train()
print("학습 완료!")

🚀 Full FT 학습 시작...


Epoch,Training Loss,Validation Loss,Lrap,Abuse Recall,Abuse F1,Clean Recall,Clean F1
1,0.384000,0.317239,0.480462,0.000000,0.000000,0.000000,0.000000
2,0.274300,0.218679,0.806250,0.078507,0.144208,0.830108,0.744096
3,0.198400,0.169710,0.861735,0.567568,0.656250,0.695699,0.753640
4,0.154700,0.153556,0.872650,0.667954,0.689701,0.687097,0.751323
5,0.139600,0.148405,0.875021,0.644788,0.686301,0.737634,0.763495


학습 완료!


In [9]:
model.save_pretrained(f"{OUTPUT_DIR}/best_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/best_model")
print("✅ 모델 저장 완료!")

eval_results = trainer.evaluate()
for k, v in eval_results.items(): print(f"  {k}: {v:.4f}")

✅ 모델 저장 완료!


  eval_loss: 0.1536
  eval_lrap: 0.8727
  eval_abuse_recall: 0.6680
  eval_abuse_f1: 0.6897
  eval_clean_recall: 0.6871
  eval_clean_f1: 0.7513
  eval_runtime: 3.6718
  eval_samples_per_second: 997.6110
  eval_steps_per_second: 15.7960
  epoch: 5.0000
